# Bike Sharing Project

Your task is to **predict how many bikes are rented in an hour** in the Capital Bikeshare system of Washington, D.C., and to find out how far **ensemble methods** improve on a single decision tree and on a linear model. In the project you combine everything you learned this week, from decision trees to boosting, voting and stacking, into one complete analysis.

The task below follows the steps of a machine learning project, in a single pass:

![The project in one pass, from top to bottom: 1. goal and metric, 2. get the data, 3. a train-test split into training data and locked test data. Steps 4 and 5 (explore, baselines) use only the training data. Steps 6 and 7 form the model selection loop: feature engineering, set hyperparameters, train model(s) and calculate the cross-validation score, then repeat. In step 8, the test data is unlocked, the models are scored on the test set and the hours with the largest errors are examined](assets/bike_sharing_project_workflow.png)

> [!NOTE]
> **AI reference solution.** This notebook is a complete solution written with Claude, for comparison. My own work is in [01_bike_sharing_project.ipynb](01_bike_sharing_project.ipynb).

# The Business Problem

The operations team of Capital Bikeshare plans its work hour by hour: how many bikes have to be available, how many trucks move bikes between stations, and how many staff are on duty. To plan, they need to know **how many bikes are rented in an hour under given conditions**: the time of day, the type of day, the season and the weather. A Tuesday at 17:00 in summer needs very different planning from a rainy Sunday morning in January.

Your model answers this question: given the conditions of an hour, it predicts its rentals. The more accurate it is, the better the team can plan, so you compare several models and choose the best one. At the end, you look at **where the model makes its biggest mistakes**, so that the team knows when they can trust its numbers and when not.

The model describes typical hours under given conditions; it is not used to forecast particular future dates. The hours you hold back as a test set can therefore be drawn **at random** from the whole period. A forecast would be a different problem: there the model has to predict hours that come after everything it was trained on, so the test set would have to be the most recent hours.

# Data Description

The data covers **two years, October 2024 to September 2026**, and ships with this repository as one table, `data/bike_sharing.csv`, with **one row per hour**. It combines three sources:

- the number of bikes rented in each hour, from the trip records that [Capital Bikeshare](https://capitalbikeshare.com/system-data) publishes. Every trip is counted in the hour in which it started;
- the weather in central Washington, D.C. in each hour, from the [Open-Meteo](https://open-meteo.com/) historical weather archive;
- the public holidays of Washington, D.C.

| Column | Source | Description |
| :-- | :-- | :-- |
| `datetime` | | start of the hour, local time in Washington, D.C. |
| `casual` | rentals | rentals by casual riders, who pay per ride or per day |
| `member` | rentals | rentals by members with an annual or monthly membership |
| `rentals` | rentals | total number of rentals in the hour, **the prediction target** |
| `temperature` | weather | air temperature in °C |
| `feels_like` | weather | apparent ("feels like") temperature in °C, which accounts for wind and humidity |
| `humidity` | weather | relative humidity in % |
| `precipitation` | weather | rain and snow in the hour, in mm of water |
| `snowfall` | weather | snowfall in the hour, in cm |
| `wind_speed` | weather | wind speed in km/h |
| `cloud_cover` | weather | share of the sky covered by clouds, in % |
| `weather_code` | weather | type of weather as a [WMO code](https://open-meteo.com/en/docs/historical-weather-api): 0 clear, 1 to 3 partly cloudy to overcast, 51 to 55 drizzle, 61 to 65 rain (light to heavy), 71 to 75 snow (light to heavy) |
| `holiday` | holidays | name of the public holiday on that day, empty on all other days |

Rentals data © Capital Bikeshare, used under the [Capital Bikeshare Data License Agreement](https://capitalbikeshare.com/data-license-agreement). Weather data by [Open-Meteo.com](https://open-meteo.com/), licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## Task

1. **Set your goal.** Decide which metric you use to judge the model, and explain why. Think about what the operations team needs to read from it.
2. **Get the data.** Load `data/bike_sharing.csv`, and take a first look at its size and columns. Turn `datetime` into a date type, so that you can extract the hour, the weekday or the month from it later.
3. **Split the data into a training and a test set.** Use `train_test_split` once. Choose the size of the test set with `test_size`, and fix the `random_state`, so that you get the same split every time you run the notebook. Keep the features and the target together: the result should be two DataFrames, `df_train` and `df_test`, that both still contain the `rentals` column. In step 7, cross-validation on the training set takes the place of a validation set. Do not look at `df_test` until step 8.
4. **Explore the training set.** How do the rentals change over the day, and is that pattern the same on working days, weekends and holidays? How do the weather columns relate to the rentals? Which columns contain the same information? Check every column carefully: **would you know its value for an hour without having counted that hour's rentals?**

> [!IMPORTANT]
> **Keep a record of every change you make to the training data by hand.** In step 8 you apply exactly the same changes to the test set. This covers, for example, new columns such as the hour of the day or a flag for working days that you extract from `datetime` and `holiday`, and every column you drop. Put these changes into a function, so that you can call the same function on the test set. Transformations inside your `Pipeline` (scaling, encoding) are applied to the test set automatically, with the values learned on the training data.

5. **Separate the features from the target, and build two baselines.** Split `df_train` into `X_train` and `y_train`; the test set stays untouched until step 8. Then build two simple models that your models have to beat:
    - a **mean baseline** with [`DummyRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html), which predicts the mean number of rentals of the training set for every hour, and
    - a **rule-based baseline** built from what you found in step 4, that a person could calculate by hand. Learn the values of the rule on the training data.
6. **Train a linear model and a single decision tree.** Train a regularised linear model, for example `Ridge`, in a `Pipeline` with the preprocessing it needs, and a `DecisionTreeRegressor`. Compare them with cross-validation: which one handles the pattern over the day better, and why? Does the tree need the same preprocessing as the linear model?
7. **Build, tune and compare ensembles with cross-validation.** Train at least:
    - a **random forest** (`RandomForestRegressor`),
    - a **gradient boosting model**, for example `XGBRegressor` or `CatBoostRegressor`: for a number, use the regressor and a regression metric. CatBoost can treat columns as categories itself (`cat_features`): find out whether that helps here,
    - and a **combination of models** with `VotingRegressor` or `StackingRegressor`.

    Tune at least one of the models with `RandomizedSearchCV` (or `GridSearchCV`), and choose the best model by its cross-validated score. **Save every model in its own variable** (and the values of your rule), so that you can evaluate all of them in the last step.
8. **Prepare the test set and evaluate all models at the very end.** First apply to `df_test` the changes you made by hand to the training data, and split it into `X_test` and `y_test`. Then calculate the **training and the test scores of every model** with your chosen metric, including both baselines, and collect them in **one table**: one row per model, with columns for the training and the test scores. Compare the models: which ones beat the baselines, how much did each step from the linear model to the ensembles gain, and how large is the difference between training and test scores?

    Finally, find out **where your best model makes its biggest mistakes**, so that the operations team knows when to trust it:
    1. **Calculate the residuals** of the test hours: actual rentals minus predicted rentals. A large positive residual means the model predicted far too few rentals, a large negative one far too many. Use only the test hours: the model has seen the training hours, so its errors there are too small to tell you anything.
    2. **Plot the residuals**: a scatter plot of the residuals against the predicted rentals, with a line at 0, and the size of the residuals by hour of the day (for example a box plot). Where does the model make its largest errors, and do the errors grow with the number of rentals?
    3. **Compare predicted and actual rentals over the day**: plot the mean actual and the mean predicted rentals for each hour of the day, separately for working days and weekends. Does the model reproduce the daily pattern you found in step 4? At which hours does it predict too many or too few rentals on average?
    4. **List the hours with the biggest mistakes**: the 10 hours with the largest positive and the 10 with the largest negative residuals, with their date, time and weather.

> [!TIP]
> If you use `CatBoostRegressor` with `cat_features` inside `cross_val_score` or a search, pass `cat_features` to `.fit()` instead of the constructor (for example `cross_val_score(..., params={"cat_features": [...]})`): set in the constructor, it breaks the copying of the model that cross-validation does.

In [6]:
# Import the libraries you need for this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform, loguniform

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, StackingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

## Step 1 – Goal and metric

**Main metric: RMSE (root mean squared error), in bikes per hour.**

The operations team plans bikes, trucks and staff hour by hour. RMSE is in the same unit as the target (bikes), so they can read it directly. Because it squares the errors before averaging, a few big misses weigh much more than many small ones, and a big miss in a busy hour (e.g. no bikes at 17:00 rush hour) costs the team more than several small misses.

I also report two more metrics for context:
- **MAE** (mean absolute error): the typical error in bikes, easy to explain. If RMSE is much larger than MAE, the model is mostly fine but sometimes misses very badly (a few large errors).
- **R²**: how much better the model is than always predicting the mean (1 = perfect, 0 = no better than the mean). It has no unit, so it is useful for comparing models but not for planning.

In [7]:
# loading the dataset
df = pd.read_csv('data/bike_sharing.csv')

# converting the 'datetime' column to datetime type
df['datetime'] = pd.to_datetime(df['datetime'])

print(f"Dataset shape: {df.shape}")
print(df.dtypes)
df.head()

Dataset shape: (17520, 13)
datetime         datetime64[us]
casual                    int64
member                    int64
rentals                   int64
temperature             float64
feels_like              float64
humidity                  int64
precipitation           float64
snowfall                float64
wind_speed              float64
cloud_cover               int64
weather_code              int64
holiday                     str
dtype: object


,datetime,casual,member,rentals,temperature,feels_like,humidity,precipitation,snowfall,wind_speed,cloud_cover,weather_code,holiday
0,2024-10-01 00:00:00,47,89,136,18.1,19.5,96,0.2,0.0,8.6,100,51,NaN
1,2024-10-01 01:00:00,18,30,48,18.2,19.6,96,0.1,0.0,8.9,100,51,NaN
2,2024-10-01 02:00:00,14,17,31,18.2,19.2,94,0.0,0.0,10.3,100,3,NaN
3,2024-10-01 03:00:00,6,9,15,17.9,18.6,93,0.0,0.0,11.7,100,3,NaN
4,2024-10-01 04:00:00,7,20,27,17.6,18.1,93,0.1,0.0,12.4,100,51,NaN


**Check:** October 2024 to September 2026 is 730 days (no 29 February in 2025 or 2026), and 730 × 24 hours = **17,520 rows**, which matches the shape. There is one row per hour, with no gaps, and the 13 columns match the data description.

## Step 3 – Split into training and test set

- **Random split:** the model describes *typical* hours under given conditions; it does not forecast future dates (see the business problem). So the test hours can be drawn at random from the whole period. For a forecast, the test set would have to be the most recent hours instead.
- **The whole DataFrame is split**, so the features and `rentals` stay together. `X` and `y` are separated later.
- `test_size=0.2`: 20 % (≈ 3,500 hours) for the final test, 80 % for training.
- `random_state=42`: the same split every time the notebook runs.

From here on, everything is done on `df_train` only. `df_test` stays locked until step 8.

In [8]:
# splitting train and test data
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

print(f"train: {df_train.shape}")
print(f"test:  {df_test.shape}")

train: (14016, 13)
test:  (3504, 13)


## Step 4 – Explore the training set

### Record of manual changes and `prepare()`

Every change to the data by hand, collected in one function, so that exactly the same changes can be applied to the test set in step 8:

1. Drop the two hours that do not exist because of daylight saving time (second Sunday of March, 02:00).
2. New columns from `datetime`: `hour`, `weekday` (0 = Monday), `month`, `year`.
3. `is_holiday` = 1 if `holiday` is not NaN, else 0.
4. `is_working_day` = 1 for Monday–Friday that is not a holiday, else 0.
5. Drop `holiday` (replaced by the flag), `feels_like` (r = 0.99 with `temperature`), `weather_code` (derived from the other weather columns), and `casual` and `member` (**leakage**).

`datetime` stays in the DataFrame so that the date of each hour can be shown in step 8; it is only left out of `X`.

Scaling and one-hot encoding are **not** done here: they learn values from the data, so they go into the `Pipeline`, which learns them from the training folds only.

In [9]:
def prepare(data):
    """All manual changes to the data, collected in one place."""
    data = data.copy()
    dt = data["datetime"]

    # 1. the hour 02:00 on the second Sunday of March does not exist (clocks go forward)
    nonexistent = (dt.dt.month == 3) & (dt.dt.dayofweek == 6) & dt.dt.day.between(8, 14) & (dt.dt.hour == 2)
    data = data[~nonexistent]
    dt = data["datetime"]

    # 2. calendar features
    data["hour"] = dt.dt.hour
    data["weekday"] = dt.dt.dayofweek
    data["month"] = dt.dt.month
    data["year"] = dt.dt.year

    # 3. + 4. flags for holidays and working days
    data["is_holiday"] = data["holiday"].notna().astype(int)
    data["is_working_day"] = ((data["weekday"] < 5) & (data["is_holiday"] == 0)).astype(int)

    # 5. drop replaced, redundant and leaking columns
    return data.drop(columns=["holiday", "feels_like", "weather_code", "casual", "member"])


df_train_prep = prepare(df_train)
print(f"rows dropped: {len(df_train) - len(df_train_prep)}")
df_train_prep.head()

rows dropped: 2


,datetime,rentals,temperature,humidity,precipitation,snowfall,wind_speed,cloud_cover,hour,weekday,month,year,is_holiday,is_working_day
17440,2026-09-27 16:00:00,1035,16.7,87,0.5,0.0,20.3,100,16,6,9,2026,0,0
16867,2026-09-03 19:00:00,1826,30.4,67,0.0,0.0,7.7,47,19,3,9,2026,0,1
14851,2026-06-11 19:00:00,1661,31.7,60,0.0,0.0,9.1,97,19,3,6,2026,0,1
5446,2025-05-15 22:00:00,1001,20.8,93,0.0,0.0,5.4,7,22,3,5,2025,0,1
6025,2025-06-09 01:00:00,114,19.7,99,0.0,0.0,3.5,100,1,0,6,2025,0,1


## Step 5 – Features, target and two baselines

`X_train` holds the features, `y_train` the target. `datetime` is left out: the model gets the hour, weekday, month and year made from it instead.

In [10]:
X_train = df_train_prep.drop(columns=["rentals", "datetime"])
y_train = df_train_prep["rentals"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print(X_train.columns.tolist())

# the same five folds for every model, so the comparison is fair
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

X_train: (14014, 12)
y_train: (14014,)
['temperature', 'humidity', 'precipitation', 'snowfall', 'wind_speed', 'cloud_cover', 'hour', 'weekday', 'month', 'year', 'is_holiday', 'is_working_day']


**Mean baseline:** predicts the mean rentals of the training set for every hour. Every model has to beat this.

In [11]:
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

cv_dummy = -cross_val_score(dummy, X_train, y_train, cv=kfold, scoring="neg_root_mean_squared_error")

print(f"predicted rentals/hour for every hour: {dummy.constant_[0][0]:.0f}")
print(f"CV RMSE: {cv_dummy.mean():.1f}")

predicted rentals/hour for every hour: 753
CV RMSE: 636.2


**Rule-based baseline:** step 4b showed that the hour of the day and the type of day decide most of the pattern. The rule: *predict the mean rentals of the same hour on the same type of day (working day or not) in the training data.* That is a table of 2 × 24 numbers a person could look up by hand.

To compare it fairly with the models, its cross-validated RMSE is calculated with the same folds: the table is learned on four folds and checked on the fifth.

In [19]:
def learn_rule(X, y):
    """Mean rentals per hour and type of day: a 24 x 2 lookup table."""
    return y.groupby([X["hour"], X["is_working_day"]]).mean()


def rule_predict(X, table):
    keys = pd.MultiIndex.from_arrays([X["hour"], X["is_working_day"]])
    return table.reindex(keys).to_numpy()


# learn the rule on the whole training set
rule_table = learn_rule(X_train, y_train)

# cross-validated RMSE of the rule, with the same folds as the models
cv_rule = []
for train_idx, val_idx in kfold.split(X_train):
    table = learn_rule(X_train.iloc[train_idx], y_train.iloc[train_idx])
    pred = rule_predict(X_train.iloc[val_idx], table)
    cv_rule.append(root_mean_squared_error(y_train.iloc[val_idx], pred))
cv_rule = np.array(cv_rule)

print(f"CV RMSE: {cv_rule.mean():.1f}")
rule_table.unstack().round().head(10)

CV RMSE: 350.4


is_working_day,0,1
hour,,
0,408.0,135.0
1,280.0,66.0
2,177.0,35.0
3,90.0,22.0
4,50.0,40.0
5,73.0,144.0
6,157.0,395.0
7,282.0,868.0
8,487.0,1470.0


The **mean baseline** is off by **636 rentals** per hour (CV RMSE). The **rule** "mean of the same hour on the same type of day" brings that down to **350**: hour and type of day alone explain most of the pattern. 
# Every model now has to beat **350**.

## Step 6 – Linear model and a single decision tree

**Preprocessing for Ridge:** `hour`, `weekday` and `month` are numbers, but for a linear model they are **categories**. As numbers, Ridge could only learn "one more hour = X more rentals", a straight line through the day, which cannot produce two peaks. One-hot encoding gives every hour its own coefficient. The other features are scaled, so that the regularisation treats all coefficients equally.

`alpha` is tuned with `RandomizedSearchCV`.
`alpha` is drawn from `loguniform(1e-3, 1e3)`: the **exponent** is drawn evenly, so every factor of 10 (0.001–0.01, 0.01–0.1, …, 100–1000) gets the same share of the 20 tries. A plain `uniform(0.001, 1000)` would put about 90 % of them between 100 and 1000 and would hardly ever test small values like the 0.39 that wins. For `alpha` (and XGBoost's `learning_rate` in step 7) the factor matters, not the difference, so the log scale is the right one.

In [20]:
categorical_features = ["hour", "weekday", "month"]
numeric_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

ridge_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", Ridge()),
])

ridge_search = RandomizedSearchCV(
    ridge_pipe,
    param_distributions={"model__alpha": loguniform(1e-3, 1e3)},
    n_iter=20,
    cv=kfold,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
)
ridge_search.fit(X_train, y_train)
best_ridge = ridge_search.best_estimator_

print(f"best alpha: {ridge_search.best_params_['model__alpha']:.4f}")
print(f"CV RMSE: {-ridge_search.best_score_:.1f}")

best alpha: 0.3905
CV RMSE: 304.4


**Why is Ridge still so far off?** It adds up one effect per feature: the coefficient for "17:00" is the same on every day, and "working day" only shifts the whole day up or down. But step 4b showed that working days and weekends have a **different shape** (two peaks vs. one hump). Ridge cannot learn that, unless it gets an **interaction**: one category for every combination of hour and type of day (48 categories).

In [ ]:
from sklearn.preprocessing import FunctionTransformer


# an extra column for Ridge only: the hour combined with the type of day, e.g. "17_1"
def add_hour_daytype(X):
    X = X.copy()
    X["hour_daytype"] = X["hour"].astype(str) + "_" + X["is_working_day"].astype(str)
    return X


preprocessor_inter = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features + ["hour_daytype"]),
])

ridge_inter_pipe = Pipeline([
    ("interaction", FunctionTransformer(add_hour_daytype)),
    ("prep", preprocessor_inter),
    ("model", Ridge()),
])

ridge_inter_search = RandomizedSearchCV(
    ridge_inter_pipe,
    param_distributions={"model__alpha": loguniform(1e-3, 1e3)},
    n_iter=20,
    cv=kfold,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
)
ridge_inter_search.fit(X_train, y_train)
best_ridge_inter = ridge_inter_search.best_estimator_

print(f"best alpha: {ridge_inter_search.best_params_['model__alpha']:.4f}")
print(f"CV RMSE: {-ridge_inter_search.best_score_:.1f}")

best alpha: 0.3905
CV RMSE: 244.0


The interaction is made inside the `Pipeline` with a `FunctionTransformer`, so it is applied to the test set automatically. It learns nothing from the data, so it could also have gone into `prepare()`; it sits in the pipeline because only Ridge needs it.

**The decision tree** gets the features as they are: no scaling and no one-hot encoding. A tree only asks questions like "is `hour` ≤ 6.5?", so the scale of a column does not matter, and it can split `hour` into any ranges it likes. It can also learn interactions by itself: a split on `is_working_day` followed by splits on `hour` gives each type of day its own daily curve.

`max_depth` and `min_samples_leaf` are tuned with `GridSearchCV`.

In [22]:
tree_grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    {"max_depth": [6, 8, 10, 12, 15, None], "min_samples_leaf": [1, 3, 5, 10, 20]},
    cv=kfold,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
tree_grid.fit(X_train, y_train)
best_tree = tree_grid.best_estimator_

print(f"best parameters: {tree_grid.best_params_}")
print(f"CV RMSE: {-tree_grid.best_score_:.1f}")

best parameters: {'max_depth': None, 'min_samples_leaf': 10}
CV RMSE: 202.0


**Comparison (CV RMSE):** Ridge 304 → Ridge with the hour × day type interaction 244 → decision tree **202**.

- **The tree handles the pattern over the day better.** It does not assume that effects simply add up: by splitting on `is_working_day` and then on `hour`, it builds a separate daily curve for each type of day, and in the same way it can learn "warm *and* dry *and* 17:00". Ridge only gets close once the interaction is built for it by hand, and it still cannot learn that, for example, rain matters more at rush hour than at night, or that the temperature curve bends above 30 °C.
- **Ridge beats the rule baseline** (304 vs. 350), because it also uses the weather and the season.
- **Preprocessing:** the tree needs **none**: no scaling (splits do not depend on the scale) and no one-hot encoding (it can cut `hour` into any ranges). Ridge needs both.
- The best tree has no depth limit but at least 10 hours per leaf (`min_samples_leaf=10`): that is the setting that stops it from memorising single hours.